# 03 — KnottedGraph vs Topoly: paper-quality Yamada scaling

This notebook benchmarks the **Yamada engines only**. For every geometric embedding, the spatial graph is converted to a PD code **outside the timed region**. KnottedGraph and Topoly then receive the same PD input.

The default **paper** profile is deliberately long-running:

- the x-axis ranges extend far beyond the earlier benchmark;
- every x-axis point uses **10 distinct deterministic embeddings**;
- timing repetitions are nested within each embedding and summarized by their median;
- the plotted point is the median across the 10 embedding-level timings;
- shaded bands are deterministic nonparametric **95% bootstrap confidence intervals** across embeddings;
- PNG figures are written at 400 dpi and PDF vector figures are written alongside them.

Timeouts are treated as censored observations and are never substituted into the confidence interval. Empirical power-law fits use only x points with at least 5 successful embeddings. The curves are empirical summaries, not proofs of Big-O scaling.


In [ ]:
from pathlib import Path
import csv, json, os, subprocess, sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install Topoly first: pip install topoly") from exc

OUT = ROOT / "User_guide" / "benchmarks"
RES = OUT / "results_latest"
FIG = OUT / "figures_latest"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("KnottedGraph:", kg_path)
print("Topoly:", Path(topoly.__file__).resolve())


## 1. Long-run configuration

A normal/local execution defaults to `PROFILE = "paper"`: 10 independent deterministic embeddings at every x value and up to 120 s **per framework per embedding**. This can therefore run for many hours near the censoring frontier.

GitHub Actions sets `CI=true`; only in that environment the notebook automatically switches to a small smoke configuration so routine CI does not launch the paper experiment. Smoke output must not be used in a paper.


In [ ]:
IS_CI = os.environ.get("CI", "").lower() == "true"

PROFILE = "smoke" if IS_CI else "paper"
EMBEDDINGS = 2 if IS_CI else 10
TIMEOUT_S = 10 if IS_CI else 120
BASE_SEED = 20260818

raw_csv = RES / "topoly_yamada_scaling_raw.csv"
aggregate_csv = RES / "topoly_yamada_scaling_aggregate.csv"

print(
    f"mode={PROFILE}, embeddings/x={EMBEDDINGS}, "
    f"timeout/framework/embedding={TIMEOUT_S}s"
)


## 2. Run the correctness-gated embedding ensemble


In [ ]:
script = ROOT / "dev" / "benchmark_topoly_extended_scaling.py"
env = dict(os.environ)
env["PYTHONPATH"] = str(SRC)
env["PYTHONNOUSERSITE"] = "1"

cmd = [
    sys.executable, str(script),
    "--profile", PROFILE,
    "--embeddings", str(EMBEDDINGS),
    "--timeout", str(TIMEOUT_S),
    "--seed", str(BASE_SEED),
]
print("Running:", " ".join(cmd))
proc = subprocess.run(
    cmd,
    cwd=ROOT,
    env=env,
    text=True,
    capture_output=True,
    timeout=None,
)
print(proc.stdout)
if proc.returncode:
    raise RuntimeError(
        f"Extended Topoly benchmark failed with exit code {proc.returncode}.\n"
        f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
    )

for line in reversed(proc.stdout.splitlines()):
    if line.startswith("SUMMARY="):
        rows = json.loads(line[8:])
        break
else:
    raise RuntimeError("Benchmark completed without SUMMARY output.")

keys = list(dict.fromkeys(key for row in rows for key in row))
with raw_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys)
    writer.writeheader()
    writer.writerows(rows)

print(f"wrote {len(rows)} embedding-level records to {raw_csv}")


## 3. Acceptance checks

These checks ensure that the expensive run actually has the intended statistical structure before any figure is accepted.


In [ ]:
from collections import defaultdict

groups = defaultdict(list)
for row in rows:
    groups[(row["family"], row["size"])].append(row)

for key, group in groups.items():
    assert len(group) == EMBEDDINGS, (key, len(group), EMBEDDINGS)
    assert len({row["embedding"] for row in group}) == EMBEDDINGS
    assert len({row["embedding_hash"] for row in group}) == EMBEDDINGS
    assert len({row["embedding_seed"] for row in group}) == EMBEDDINGS

for row in rows:
    if row["correctness"] == "PASS":
        assert row["knottedgraph_status"] == "ok"
        assert row["topoly_status"] == "ok"
        assert row["pd_hash"]

print(f"PASS: every evaluated x point contains {EMBEDDINGS} distinct embeddings.")
print("PASS: all paired successful framework evaluations passed Laurent-polynomial equivalence.")


## 4. Generate paper-quality figures and confidence intervals

The plotting script can be rerun later from the saved raw CSV without rerunning the expensive timing experiment. Confidence intervals are bootstrap intervals over the independent embedding-level observations, not over repeated stopwatch calls.


In [ ]:
plot_script = ROOT / "dev" / "plot_topoly_scaling.py"
plot_proc = subprocess.run(
    [
        sys.executable, str(plot_script), str(raw_csv),
        "--figure-dir", str(FIG),
        "--aggregate-csv", str(aggregate_csv),
    ],
    cwd=ROOT,
    env=env,
    text=True,
    capture_output=True,
)
print(plot_proc.stdout)
if plot_proc.returncode:
    raise RuntimeError(
        f"Plot generation failed.\nSTDOUT:\n{plot_proc.stdout}\nSTDERR:\n{plot_proc.stderr}"
    )

expected_stems = [
    "topoly_vs_knottedgraph_crossings_fixed",
    "topoly_vs_knottedgraph_crossings_throughput",
    "topoly_vs_knottedgraph_edges",
    "topoly_vs_knottedgraph_vertices_k4",
    "topoly_vs_knottedgraph_prism_V",
    "topoly_vs_knottedgraph_prism_E",
]
for stem in expected_stems:
    assert (FIG / f"{stem}.png").exists()
    assert (FIG / f"{stem}.pdf").exists()
assert aggregate_csv.exists()
print("PASS: all figure PNG/PDF pairs and aggregate confidence-interval CSV were created.")


## 5. Statistical interpretation

For each x-axis point and framework:

1. each of the 10 geometric embeddings is generated from a documented deterministic seed;
2. both frameworks evaluate the same PD code for that embedding;
3. multiple stopwatch repetitions, when used, are reduced to one median timing for that embedding;
4. the 10 resulting embedding-level timings form the statistical sample;
5. the plotted center is their median and the shaded region is a 95% bootstrap confidence interval for that median.

This avoids treating repeated timings of one geometry as independent observations. Points for which some embeddings time out use only successful observations for descriptive plotting; fully censored points are marked separately. Scaling fits require at least five successful embeddings at an x value.
